**IMPORTAÇÃO E EXPLORAÇÃO DOS DADOS**

In [44]:
#Importação de Arquivos
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
#Ler arquivo de Voos
df = pd.read_csv('flights_delays_120.csv')
df.head()

,airline,origin,destination,departure_hour,day_of_week,weather,delayed
0,TravelAir,GIG,FOR,11,5,Storm,0
1,JetCloud,CNF,SSA,11,3,Wind,0
2,SkyWings,POA,SSA,4,5,Fog,1
3,JetCloud,BSB,FOR,6,4,Storm,1
4,JetCloud,GRU,FOR,3,1,Rain,1


In [5]:
#Informações sobre a base
df.describe()

,departure_hour,day_of_week,delayed
count,120.000000,120.000000,120.000000
mean,11.258333,3.891667,0.433333
std,7.300853,1.999142,0.497613
min,0.000000,1.000000,0.000000
25%,4.000000,2.000000,0.000000
50%,11.000000,4.000000,0.000000
75%,18.000000,5.000000,1.000000
max,23.000000,7.000000,1.000000


In [7]:
#Informações sobre os tipos das colunas
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   airline         120 non-null    object
 1   origin          120 non-null    object
 2   destination     120 non-null    object
 3   departure_hour  120 non-null    int64 
 4   day_of_week     120 non-null    int64 
 5   weather         120 non-null    object
 6   delayed         120 non-null    int64 
dtypes: int64(3), object(4)
memory usage: 6.7+ KB


In [10]:
#Informações sobre a quantidade de linhas e colunas
df.shape

(120, 7)

**PREPARAÇÃO E PRÉ-PROCESSAMENTO DOS DADOS**

In [18]:
#Separando variáveis
df_preditoras = df.drop(['delayed'], axis=1)
df_alvo = df['delayed']

In [19]:
#Verificando resultados
df_preditoras.head()

,airline,origin,destination,departure_hour,day_of_week,weather
0,TravelAir,GIG,FOR,11,5,Storm
1,JetCloud,CNF,SSA,11,3,Wind
2,SkyWings,POA,SSA,4,5,Fog
3,JetCloud,BSB,FOR,6,4,Storm
4,JetCloud,GRU,FOR,3,1,Rain


In [20]:
#Verificando resultados
df_alvo.head()

,delayed
0,0
1,0
2,1
3,1
4,1


In [22]:
#Tratamento de Varáveis
df_tratado = pd.get_dummies(df_preditoras)
df_tratado.head()

,departure_hour,day_of_week,airline_AirOne,airline_FlyFast,airline_JetCloud,airline_SkyWings,airline_TravelAir,origin_BSB,origin_CNF,origin_GIG,...,destination_BEL,destination_CWB,destination_FOR,destination_REC,destination_SSA,weather_Clear,weather_Fog,weather_Rain,weather_Storm,weather_Wind
0,11,5,False,False,False,False,True,False,False,True,...,False,False,True,False,False,False,False,False,True,False
1,11,3,False,False,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,4,5,False,False,False,True,False,False,False,False,...,False,False,False,False,True,False,True,False,False,False
3,6,4,False,False,True,False,False,True,False,False,...,False,False,True,False,False,False,False,False,True,False
4,3,1,False,False,True,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False


In [23]:
#Transformando False e True em 0 e 1
df_tratado = df_tratado.astype(int)
df_tratado.head()

,departure_hour,day_of_week,airline_AirOne,airline_FlyFast,airline_JetCloud,airline_SkyWings,airline_TravelAir,origin_BSB,origin_CNF,origin_GIG,...,destination_BEL,destination_CWB,destination_FOR,destination_REC,destination_SSA,weather_Clear,weather_Fog,weather_Rain,weather_Storm,weather_Wind
0,11,5,0,0,0,0,1,0,0,1,...,0,0,1,0,0,0,0,0,1,0
1,11,3,0,0,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,1
2,4,5,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0
3,6,4,0,0,1,0,0,1,0,0,...,0,0,1,0,0,0,0,0,1,0
4,3,1,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0


In [36]:
#Divisão dos Dados
x_treino, x_teste, y_treino, y_teste = train_test_split(df_tratado, df_alvo, test_size=0.3, random_state=42, stratify=df_alvo)


**TREINAMENTO DO MODELO XGBOOST**

In [38]:
#Criar Instância do modelo
modelo = XGBClassifier()

In [39]:
#Treina o modelo com dados de treino
modelo.fit(x_treino, y_treino)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

**AVALIAÇÃO DO MODELO**

In [41]:
#Realiza Predições e Avalia
y_pred = modelo.predict(x_teste)

print("Primeira 5 predições do modelo: ", y_pred[:10])
print("Gabarito Real: ", y_teste[:10].values)

Primeira 5 predições do modelo:  [0 0 1 1 0 0 0 0 1 0]
Gabarito Real:  [0 0 0 1 0 0 0 0 1 0]


In [45]:
#Avalia Acurácia
acuracia = accuracy_score(y_teste, y_pred)
print(f"Acurácia do Modelo: {acuracia * 100:.2f}%")

Acurácia do Modelo: 91.67%


In [46]:
#Relatório de Classificação
print(classification_report(y_teste, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.90      0.92        20
           1       0.88      0.94      0.91        16

    accuracy                           0.92        36
   macro avg       0.91      0.92      0.92        36
weighted avg       0.92      0.92      0.92        36



In [47]:
#Matriz de Confusão
print(confusion_matrix(y_teste, y_pred))

[[18  2]
 [ 1 15]]


Podemos verificar que o número de falos positvos ou negativos são baixíssimos.